In [ ]:
import os
import re
import pandas as pd
from PIL import Image
from tqdm import tqdm
from google import genai
import time
import random
from io import BytesIO


API_KEY = "ENTER API KEY HERE" # Replace with your actual API key


Settings

In [2]:
IMAGE_FOLDER = f"../../image_similarities/pictures/"

OUT_BATHROOM = f"{IMAGE_FOLDER}bathroom_objects.csv"
OUT_KITCHEN = f"{IMAGE_FOLDER}kitchen_objects.csv"


client = genai.Client(api_key=API_KEY)

PROMPT = """
List ALL visible objects in this image from left to right.

Rules:
- Return only object names.
- Use singular nouns.
- Do not describe relationships.
- Do not return full sentences.
- Include furniture, appliances, fixtures and decorations.

Return format:

chair, table, window, lamp
"""


Helpers

In [ ]:
def extract_subject(filename):
    """
    Example:
    102_kit.jpg -> 102
    102_kit_gen1.png -> 102
    """
    m = re.match(r"(\d+)", filename)
    return m.group(1) if m else filename

def get_objects(image_path,
                max_retries=15,
                initial_wait=5,
                resize=False):

    # Load
    image = Image.open(image_path)
    image = image.convert("RGB")

    if resize:
        image.thumbnail((1024, 768))

    for attempt in range(max_retries):

        try:

            response = client.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    PROMPT,
                    image
                ]
            )

            text = response.text.strip()

            objects = [
                obj.strip().lower()
                for obj in text.split(",")
                if obj.strip()
            ]

            # remove duplicates
            seen = set()
            objects = [x for x in objects if not (x in seen or seen.add(x))]

            return objects

        except Exception as e:

            print(f"Attempt {attempt+1}/{max_retries} failed:")
            print(e)

            if attempt == max_retries - 1:
                raise

            wait_time = initial_wait * (2 * attempt)

            # small random jitter
            wait_time += random.uniform(0, 2)

            print(f"Retrying in {wait_time:.1f} sec...")
            time.sleep(wait_time)

def save_csv(subject_dict, outfile, sort_keys=True):

    rows = []

    keys = list(subject_dict.keys())

    if sort_keys:
        keys = sorted(keys)

    for key in keys:
        objects = sorted(subject_dict[key])
        rows.append([key] + objects)

    if len(rows) == 0:
        return

    max_objects = max(len(r) - 1 for r in rows)

    columns = ["id"] + [f"obj{i+1}" for i in range(max_objects)]

    padded_rows = [
        row + [""] * (len(columns) - len(row))
        for row in rows
    ]

    df = pd.DataFrame(padded_rows, columns=columns)
    df.to_csv(outfile, index=False)



Process

In [ ]:
bathroom_dict = {}
kitchen_dict = {}

image_files = []

for root, dirs, files in os.walk(IMAGE_FOLDER):
    for f in files:
        if (
            f.lower().endswith((".jpg", ".jpeg", ".png"))
            and "canceled" not in root.lower()
            and "excluded" not in root.lower()
        ):
            image_files.append(os.path.join(root, f))

for file in tqdm(image_files):

    subject = extract_subject(file)

    try:
        objects = get_objects(file)

        # Bathroom
        if "_bat" in file.lower():

            if subject not in bathroom_dict:
                bathroom_dict[subject] = set()

            bathroom_dict[subject].update(objects)

            # Immediately update CSV
            save_csv(bathroom_dict, OUT_BATHROOM)

        # Kitchen
        elif "_kit" in file.lower():

            if subject not in kitchen_dict:
                kitchen_dict[subject] = set()

            kitchen_dict[subject].update(objects)

            # Immediately update CSV
            save_csv(kitchen_dict, OUT_KITCHEN)

        print(f"✓ {os.path.basename(file)}")

    except Exception as e:
        print(f"Error with {file}: {e}")

print("Finished.")

  0%|          | 0/85 [00:00<?, ?it/s]

Attempt 1/15 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 6.9 sec...


  1%|          | 1/85 [00:40<56:28, 40.34s/it]

✅ Done.


  2%|▏         | 2/85 [00:58<37:58, 27.46s/it]

...

✅ Done.
Attempt 1/15 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 6.2 sec...
Attempt 2/15 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 10.0 sec...


 41%|████      | 35/85 [29:38<42:14, 50.68s/it]

✅ Done.


c:\Users\JLU-SU\miniconda3\envs\drwSim\lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (94080000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Attempt 1/15 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complete.', 'status': 'UNAVAILABLE'}}
Retrying in 5.7 sec...
Attempt 2/15 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complete.', 'status': 'UNAVAILABLE'}}
Retrying in 10.4 sec...
Attempt 3/15 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complete.', 'status': 'UNAVAILABLE'}}
Retrying in 20.5 sec...
Attempt 4/15 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complete.', 'status': 'UNAVAILABLE'}}
Retrying in 41.2 sec...
Attempt 5/15 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complete.', 'status': 'UNAVAILABLE'}}
Retrying in 80.4 sec...
Attempt 6/15 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'Deadline expired before operation could complet

In [ ]:
# SAVE CSV

save_csv(bathroom_dict, OUT_BATHROOM)
save_csv(kitchen_dict, OUT_KITCHEN)

print("Done.")

Get Objects in stimuli

In [ ]:
from PIL import Image
import os

IMAGE_FOLDER = r"../../stimuli/bathroom"
OUT_BATHROOM = "../../stimuli/bathroom/bathroom_objects.csv"

# Bathroom stimuli

bathroom_dict = {}

image_files = [
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith((".png"))
]


for file in tqdm(image_files):

    full_path = os.path.join(IMAGE_FOLDER, file)

    try:
        objects = get_objects(full_path, resize=True)
        bathroom_dict[file] = set()
        bathroom_dict[file].update(objects)

    except Exception as e:
        print(f"Error with {file}: {e}")

# SAVE CSV

save_csv(bathroom_dict, OUT_BATHROOM)


# Kitchen stimuli

IMAGE_FOLDER = r"../../stimuli/kitchen"
OUT_KITCHEN = "../../stimuli/kitchen/kitchen_objects.csv"

kitchen_dict = {}

image_files = [
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith((".png"))
]


for file in tqdm(image_files):

    full_path = os.path.join(IMAGE_FOLDER, file)
    try:
        objects = get_objects(full_path, resize=True)
        kitchen_dict[file] = set()
        kitchen_dict[file].update(objects)


    except Exception as e:
        print(f"Error with {file}: {e}")


save_csv(kitchen_dict, OUT_KITCHEN)

print("Done.")



100%|██████████| 50/50 [01:20<00:00,  1.60s/it]


TypeError: save_csv() got an unexpected keyword argument 'sort_subjects'

In [6]:

# Kitchen stimuli

IMAGE_FOLDER = r"../../stimuli/kitchen"
OUT_KITCHEN = "../../stimuli/kitchen/kitchen_objects.csv"

kitchen_dict = {}

image_files = [
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith((".png"))
]


for file in tqdm(image_files):

    full_path = os.path.join(IMAGE_FOLDER, file)
    try:
        objects = get_objects(full_path, resize=True)
        kitchen_dict[file] = set()
        kitchen_dict[file].update(objects)


    except Exception as e:
        print(f"Error with {file}: {e}")


save_csv(kitchen_dict, OUT_KITCHEN)

print("Done.")

  8%|▊         | 4/50 [00:06<01:11,  1.55s/it]

Attempt 1/5 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 5.6 sec...
Attempt 2/5 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 10.1 sec...


100%|██████████| 50/50 [04:33<00:00,  5.48s/it]

Done.
